# Figure 5 | A novel lncRNA frontal cortex sQTL colocalizes with neuroticism GWAS

**a.** Regional association plots at chr3:35.55-35.70 Mb for neuroticism (UK Biobank) and molecular QTLs in frontal cortex (BA9). Tracks show −log10(P) for neuroticism; novel lncRNA ENSG00000295459 sQTLs at junctions J1 (chr3:35638953:35639754:clu_37423_-) and J3 (chr3:35639634:35639754:clu_37423_-); novel lncRNA ENSG00000295459 eQTL; and neighboring protein-coding gene ARPP21 eQTL. Points are colored by LD (r²) with the colocalization (CoS) lead variant (purple diamond; chr3:35639573:G:A). Gene models (bottom) show ARPP21 and ENSG00000295459 genomic context. The neuroticism locus aligns with the lncRNA sQTLs, but not with ARPP21 or lncRNA eQTLs.

**b.** Zoom of the shaded interval in a, showing ARPP21 exons (+ strand), ENSG00000295459 transcript models (− strand), and splice junctions J1, J2, and J3. Black segments mark a Ribo-seq-supported ncORF with arrows indicating ncORF start codons. The CoS lead variant lies 61 bp away from the J3 splice acceptor.

**c.** STAR splice-junction counts for J1 and J3 by genotype at the CoS lead variant (G/G, n=93; G/A, n=132; A/A, n=43). The A allele is associated with higher usage of canonical junction J1 and lower usage of cryptic junction J3, consistent with an sQTL effect shifting splicing toward the annotated acceptor and away from the cryptic acceptor.

**d.** LocusCompare plots of neuroticism −log10(P) versus −log10(P) for ENSG00000295459 sQTL J1, sQTL J3, and eQTL, and ARPP21 eQTL. Points are colored by LD with the CoS lead as in (a). neuroticism associations colocalize with the lncRNA sQTLs (CoSnpc = 0.98 for both junctions) but not the ARPP21 or lncRNA eQTL.

**e.** Sample-level TPM of ENSG00000295459 across organs in adult GTEx (top) and developmental GTEx (dGTEx, bottom). Box plots show median and IQR; whiskers, 1.5× IQR. ENSG00000295459 is expressed in the brain across development.


## Setup


In [ ]:
import gzip
import json
import lzma
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon
import numpy as np
import pandas as pd
import pysam
import seaborn as sns
from IPython.display import display
from tqdm.auto import tqdm
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.transforms import blended_transform_factory
from matplotlib.collections import LineCollection
import matplotlib.colors as mcolors

plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=6)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("legend", title_fontsize=6)
plt.rc("font", size=6)
plt.rc("figure", titlesize=7)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.figsize"] = (7.2, 5.0)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

# Locus, phenotypes and plot parameters
tissue = "Brain_Frontal_Cortex_BA9"
region_id = "chr3_34773315_37490365"
cos_lead_variant = "chr3:35639573:G:A"
_ref, _alt = cos_lead_variant.split(":")[2:]
gt_order = [_ref + _ref, _ref + _alt, _alt + _alt]  # GG, GA, AA
# STAR SJ.out.tab coords (J1 annotated / J3 cryptic; last flag = primary ↑ with A)
sashimi_junctions = [
    ("J1", 35638954, 35639753, True),
    ("J3", 35639635, 35639753, False),
]
ba9_smtsd = "Brain - Frontal Cortex (BA9)"
gwas_primary = "UKB_20127_Neuroticism_score"
gene_ids = ["ENSG00000295459.1"]
locuscompare_extra = [
    {"gene_id": "ENSG00000172995.17", "qtl_kind": "eqtl"},
]
gene_name_override = {"ENSG00000295459": "ENSG00000295459"}

junction_label_by_intron = {
    (35638953, 35639754): "J1",
    (35639045, 35639754): "J2",
    (35639634, 35639754): "J3",
}
focal_transcripts = [
    {
        "transcript_id": "ENST00000730226.1",
        "label": "ENST00000730226",
        "exons": [(35638662, 35638953), (35639754, 35640182)],
        "junctions": [
            {"coords": (35638953, 35639754), "label": "J1"},
            {"coords": (35639634, 35639754), "label": "J3"},
        ],
        # Ribo-seq translated ncORF (riboSeqMin_0.6_v47ndDecoy); − strand start codon
        "ncorf_exons": [(35638850, 35638953), (35639754, 35640154)],
        "start_codon": "CTG",
        "start_pos": 35640154,
    },
    {
        "transcript_id": "ENST00000730227.1",
        "label": "ENST00000730227",
        "exons": [(35638671, 35639045), (35639754, 35640178)],
        "junctions": [
            {"coords": (35639045, 35639754), "label": "J2"},
        ],
        "ncorf_exons": [(35638850, 35639045), (35639754, 35639825)],
        "start_codon": "ATG",
        "start_pos": 35639825,
    },
]
ncorf_intervals = {"ENSG00000295459": [(35638850, 35640154)]}
sqtl_phenotypes = [
    "chr3:35638953:35639754:clu_37423_-:ENSG00000295459.1",
    "chr3:35639634:35639754:clu_37423_-:ENSG00000295459.1",
]
focal_zoom_win = (35_638_500, 35_640_500)
locus_win_half = 100_000
zoom_win_half = 25_000

ld_bounds = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
ld_colors = ["black", "#4298B5", "#6FA287", "#E98300", "#B83A4B"]
ld_cmap = ListedColormap(ld_colors)
ld_norm = BoundaryNorm(ld_bounds, ld_cmap.N)
ld_default_color = "#1a1a1a"
lead_marker_color = "#734675"
lead_marker_s = 7
base_marker_s = 2
gene_type_palette = {
    "protein_coding": sns.color_palette("Paired", 12)[1],
    "lncRNA": sns.color_palette("Paired", 12)[5],
    "novel_lncRNA": sns.color_palette("Paired", 12)[5],
    "existing_lncRNA": sns.color_palette("Paired", 12)[4],
    "gwas": "#888888",
}

# Zenodo ColocBoost helpers
def tissue_from_cos_id(cid):
    s = str(cid)
    i = s.rfind("_")
    return s[:i] if i > 0 else s


def lead_from_lists(variants, vcp):
    vs = list(variants)
    vcps = list(vcp)
    ps = pd.to_numeric(pd.Series(vcps), errors="coerce")
    if len(ps) == len(vs) and ps.notna().any():
        return vs[int(ps.values.argmax())]
    return vs[0]


def cos_to_long(row):
    vs = list(row["variants"])
    ps = list(row["vcp"])
    return pd.DataFrame({
        "variant_id": vs,
        "vcp": ps[: len(vs)] + [float("nan")] * (len(vs) - len(ps)),
    })


def _coloc_list_len(val):
    if val is None:
        return 0
    if isinstance(val, float) and np.isnan(val):
        return 0
    if isinstance(val, np.ndarray):
        return len(val)
    if isinstance(val, (list, tuple, set)):
        return len(val)
    return 0


def _typeset(vals):
    if vals is None:
        return set()
    if isinstance(vals, float) and np.isnan(vals):
        return set()
    if isinstance(vals, np.ndarray):
        items = vals.tolist()
    elif isinstance(vals, (list, tuple, set)):
        items = list(vals)
    else:
        items = [vals]
    out = set()
    for v in items:
        if v is None:
            continue
        s = str(v).strip()
        if s and s not in ("nan", "None", "<NA>"):
            out.add(s)
    return out


def _member_qtl_class(types):
    t = _typeset(types)
    lnc = t & {"novel_lncRNA", "existing_lncRNA"}
    non = t - {"novel_lncRNA", "existing_lncRNA"}
    if not lnc:
        return "non-lncRNA"
    if non:
        return "Shared"
    return "lncRNA-specific (robust)"


def _locus_qtl_label(member_classes):
    labs = set(member_classes)
    if labs == {"lncRNA-specific (robust)"}:
        return "lncRNA-specific (robust)"
    if "Shared" in labs or ("non-lncRNA" in labs and "lncRNA-specific (robust)" in labs):
        return "Shared"
    return "non-lncRNA"


def load_main_colocboost(colocboost_dir, *, cols=None):
    frames = []
    for path in sorted(colocboost_dir.glob("*.ColocBoost.parquet")):
        if path.name.startswith("cross_"):
            continue
        tissue = path.name.replace(".ColocBoost.parquet", "")
        df = pd.read_parquet(path, columns=cols)
        df["tissue"] = tissue
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def build_cross_tissue_members(cross_tissue_loci_path, colocboost_dir, *, include_ucos=False):
    loci = pd.read_parquet(cross_tissue_loci_path)
    all_cos = load_main_colocboost(colocboost_dir).set_index("cos_id", drop=False)
    memb = loci.explode("member_cos_ids").rename(columns={"member_cos_ids": "cos_id"})
    memb["cos_id"] = memb["cos_id"].astype(str)
    cos_rows = memb[["locus_id", "cos_id"]].merge(
        all_cos.reset_index(drop=True), on="cos_id", how="inner"
    )
    cos_rows["lead_variant_id"] = [
        lead_from_lists(v, p) for v, p in zip(cos_rows["variants"], cos_rows["vcp"])
    ]
    if "phenotype_id" not in cos_rows.columns:
        cos_rows["phenotype_id"] = cos_rows["phenotype_ids"]

    qtl_by_locus = (
        cos_rows.groupby("locus_id")["gene_types_robust"]
        .agg(lambda s: _locus_qtl_label(_member_qtl_class(x) for x in s))
    )
    locus_summary = loci.copy()
    locus_summary["qtl_label"] = locus_summary["locus_id"].map(qtl_by_locus)

    if not include_ucos:
        return locus_summary, cos_rows

    member_ids = set(memb["cos_id"])
    has_gwas = all_cos["gwas_traits"].map(_coloc_list_len) > 0
    has_qtl = all_cos["gene_ids_robust"].map(_coloc_list_len) > 0
    ucos = all_cos.loc[
        ~all_cos.index.isin(member_ids) & (has_gwas ^ has_qtl)
    ].reset_index(drop=True)
    ucos = ucos.loc[has_gwas.reindex(ucos["cos_id"]).fillna(False).values].copy()
    ucos["locus_id"] = "ucos_" + ucos["cos_id"].astype(str)
    ucos["lead_variant_id"] = [
        lead_from_lists(v, p) for v, p in zip(ucos["variants"], ucos["vcp"])
    ]
    if "phenotype_id" not in ucos.columns:
        ucos["phenotype_id"] = ucos["phenotype_ids"]
    members = pd.concat([cos_rows, ucos], ignore_index=True)
    return locus_summary, members



## Paths

In [ ]:
annot_feather = Path("path/to/gencode.v47.annotation.gtf.feather")
gencode_class = Path("path/to/gencode_v39_v47_gene_classification.txt")
trait_abbr_tsv = Path("path/to/table_s2_abbreviations.txt")
gwas_meta_file = Path("path/to/barbeira_full_gwas_metadata.txt")
gtex_colors_json = Path("refs/colors.json")
gtex_sample_meta_file = Path("path/to/GTEx_Analysis_v10_Annotations_SampleAttributesDS.txt")
gtex_tpm = Path("path/to/GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_tpm.parquet")
dgtex_colors_file = Path("path/to/dgtex_colors_final.csv")
dgtex_meta_file = Path("path/to/dGTEx_Analysis_2025-07-21_v1_RNA_Metadata_subset_EXCLUSIONS.xlsx")
dgtex_tpm = Path("path/to/dGTEx_Analysis_2025-12-10_v1_unstranded_collapsed_trimmed_RNASeQCv2.4.3_gene_tpm.parquet")

eqtl_nominal_dir = Path("path/to/v11_eqtl_allpairs")
sqtl_nominal_dir = Path("path/to/v11_sqtl_allpairs")
gwas_nominal_dir = Path("path/to/imputed_gwas")
ld_dir = Path("path/to/ld_blocks_gtex_eur")
# Zenodo coloc tables (CoS / VCP). Association −log10(p) tracks use portal/GWAS nominals below.
tissue_coloc_path = Path(f"path/to/{tissue}.ColocBoost.parquet")
# Protected / dbGaP inputs for panel-c STAR SJ violins.
sashimi_vcf = Path(
    "path/to/GTEx_Analysis_2021-02-11_v9_WholeGenomeSeq_953Indiv.SHAPEIT2_phased.vcf.gz"
)
sashimi_sj_dir = Path("path/to/gtex_star_sj")

fig_dir = Path("figures")


## Load data


### Annotations and trait labels


In [ ]:
s2_abbr = set()
with open(trait_abbr_tsv) as fh:
    fh.readline()
    for line in fh:
        parts = line.strip().split()
        if parts:
            s2_abbr.add(parts[-2])

gwas_meta = pd.read_csv(gwas_meta_file, sep="\t")
trait_labels = {}
for tag, code in zip(gwas_meta["Tag"], gwas_meta["new_abbreviation"]):
    if pd.isna(code) or code not in s2_abbr:
        continue
    trait_labels[str(tag)] = str(code)


def gwas_short_label(gwas_id):
    if str(gwas_id) == gwas_primary:
        return "Neuroticism (UKBB)"
    return trait_labels.get(str(gwas_id), str(gwas_id).replace("_", " ")[:40])


ann = pd.read_feather(annot_feather)
gene_ann = ann[ann["Feature"] == "gene"][
    ["gene_id", "gene_name", "gene_type", "Chromosome", "Start", "End", "Strand"]
].drop_duplicates("gene_id")
gene_ann["stable"] = gene_ann["gene_id"].str.split(".").str[0]
stable_to_name = dict(zip(gene_ann["stable"], gene_ann["gene_name"]))
stable_to_type = dict(zip(gene_ann["stable"], gene_ann["gene_type"]))

gclass = pd.read_table(gencode_class, low_memory=False)
gclass["stable"] = gclass["gene_id_stable"].astype(str)
split_map = dict(zip(gclass["stable"], gclass["gene_type_grouped"]))

for _stable, _name in gene_name_override.items():
    stable_to_name[_stable] = _name

print("Trait labels:", len(trait_labels))
print("Genes:", {gid: stable_to_name.get(gid.split(".")[0], gid) for gid in gene_ids})
print("Tissue:", tissue, "region:", region_id)

### Coloc event (Zenodo tissue leads + long)


In [ ]:
def _parse_neglog10_list(val):
    out = []
    for tok in str(val or "").replace(";", ",").split(","):
        tok = tok.strip()
        if not tok or tok.lower() in ("nan", "none", "<na>"):
            out.append(float("nan"))
            continue
        try:
            out.append(float(tok))
        except ValueError:
            out.append(float("nan"))
    return out


leads = pd.read_parquet(tissue_coloc_path)
long = pd.DataFrame()
ph = "phenotype_ids"

gene_core = gene_ids[0].split(".")[0]
_var_hit = leads["variants"].map(lambda vs: cos_lead_variant in list(vs))
_pheno_hit = leads[ph].astype(str).str.contains(gene_core, regex=False) & leads[ph].astype(str).str.contains(
    gwas_primary, regex=False
)
_focal = leads[_pheno_hit & (_var_hit | _pheno_hit)].copy()
if _focal.empty:
    _focal = leads[_pheno_hit].copy()
print("Focal gene×GWAS CoS rows:", len(_focal))
if _focal.empty:
    raise ValueError(
        f"No focal CoS for {gene_ids[0]} × {gwas_primary} in {tissue_coloc_path}"
    )

_show_cols = [c for c in ["cos_id", ph, "cos_npc", "variants", "vcp"] if c in _focal.columns]
_focal = _focal.copy()
_focal["_max_vcp"] = [
    float(pd.to_numeric(pd.Series(list(v)), errors="coerce").max())
    for v in _focal["vcp"]
]
display(_focal[_show_cols + ["_max_vcp"]].sort_values("_max_vcp", ascending=False).head(5))
focal_row = _focal.sort_values("_max_vcp", ascending=False).iloc[0]
focal_cos_id = str(focal_row["cos_id"])
_vars = list(focal_row["variants"])
if not (_vars and cos_lead_variant in _vars):
    _by_var = leads[leads["variants"].map(lambda vs: cos_lead_variant in list(vs))]
    if len(_by_var):
        focal_row = _by_var.iloc[0]
        focal_cos_id = str(focal_row["cos_id"])

lead_pos = int(cos_lead_variant.split(":")[1])
lead_panel_id = cos_lead_variant.replace(":", "_") + "_b38"

cos_long = cos_to_long(focal_row)
if cos_long.empty:
    raise ValueError(f"No variant/VCP rows for focal CoS {focal_cos_id}")
full_df = cos_long.copy()
full_df["cos_id"] = focal_cos_id
full_df["phenotype_id"] = focal_row[ph]
full_df["cos_npc"] = focal_row["cos_npc"]
if "npc_outcomes" in focal_row.index:
    full_df["npc_outcomes"] = focal_row["npc_outcomes"]

_lead_hit = cos_long[cos_long["variant_id"].astype(str) == cos_lead_variant]
_lead_vcp = float(pd.to_numeric(_lead_hit["vcp"], errors="coerce").max()) if len(_lead_hit) else float("nan")
_cos_npc = float(focal_row["cos_npc"]) if pd.notna(focal_row.get("cos_npc")) else float("nan")

_pval_col = "cos_max_pval_nominal_phenotypes"
_gwas_logp = float("nan")
parts = [p.strip() for p in str(focal_row[ph]).split(";") if p.strip()]
vals = _parse_neglog10_list(focal_row[_pval_col]) if _pval_col in focal_row.index else []
for i, part in enumerate(parts):
    if gwas_primary in part and i < len(vals):
        _gwas_logp = float(vals[i])
        break
print(
    f"Lead {cos_lead_variant}: GWAS CoS-max -log10(p)={_gwas_logp:.4f}  "
    f"(p≈{10**(-_gwas_logp):.3e}); VCP={_lead_vcp:.4f}; "
    f"cos_npc={_cos_npc:.4f}"
)
print("(Association tracks use GTEx/GWAS nominals; VCP track uses ColocBoost variant lists.)")

_orf = ncorf_intervals.get(gene_ids[0].split(".")[0], [])
_in_orf = any(lo <= lead_pos <= hi for lo, hi in _orf)
print(f"Lead in contiguous ncORF {_orf}: {_in_orf}")
print(f"Focal CoS: {focal_cos_id}")


### Locus associations


In [ ]:
def _log(msg):
    print(msg, flush=True)


def parse_variant_full(vid):
    parts = str(vid).split(":")
    return parts[0], int(parts[1]), parts[2], parts[3]


def coloc_to_panel(vid):
    chrom, pos, ref, alt = parse_variant_full(vid)
    return f"{chrom}_{pos}_{ref}_{alt}_b38"


def gene_display_name(gid):
    core = str(gid).split(".")[0]
    return stable_to_name.get(core, core)


def gene_biotype_key(gid):
    core = str(gid).split(".")[0]
    split = split_map.get(core)
    if isinstance(split, str) and split and split != "protein_coding":
        return split
    gt = stable_to_type.get(core, "lncRNA")
    if gt == "protein_coding":
        return "protein_coding"
    return "lncRNA"


_ld_block_index = {}


def _variant_to_ld_id(variant_id):
    return str(variant_id).replace(":", "_")


def _resolve_ld_matrix_id(variant_id, ld_ids):
    ld_set = set(ld_ids)
    if ":" in str(variant_id):
        base = _variant_to_ld_id(variant_id)
        candidates = (base, f"{base}_b38", coloc_to_panel(variant_id))
    else:
        candidates = (str(variant_id),)
    for cand in candidates:
        if cand in ld_set:
            return cand
    if ":" in str(variant_id):
        chrom, pos, _, _ = parse_variant_full(variant_id)
        prefix = f"{chrom}_{pos}_"
        hits = sorted(v for v in ld_ids if v.startswith(prefix))
        if hits:
            return hits[0]
    return None


def _build_ld_block_index(chrom):
    chrom = chrom if str(chrom).startswith("chr") else f"chr{chrom}"
    rows = []
    for bim_path in sorted(ld_dir.glob(f"LD_{chrom}_*.bim")):
        block_id = bim_path.name.replace(".bim", "").replace("LD_", "")
        bim = pd.read_table(bim_path, header=None, usecols=[3], names=["pos"])
        rows.append((int(bim["pos"].min()), int(bim["pos"].max()), block_id))
    _ld_block_index[chrom] = sorted(rows)


def _ld_block_id_for_position(chrom, pos):
    chrom = chrom if str(chrom).startswith("chr") else f"chr{chrom}"
    if chrom not in _ld_block_index:
        _build_ld_block_index(chrom)
    for start, end, block_id in _ld_block_index[chrom]:
        if start <= pos <= end:
            return block_id
    raise ValueError(f"No LD block found for {chrom}:{pos}")


def _load_ld_r2_to_ref(ld_block_id, ld_ref_id, win_lo, win_hi):
    bim = pd.read_table(
        ld_dir / f"LD_{ld_block_id}.bim", header=None,
        names=["chr", "variant_id", "cm", "pos", "a1", "a2"],
    )
    ref_hits = bim.index[bim["variant_id"] == ld_ref_id]
    if not len(ref_hits):
        return pd.Series(dtype=float)
    ref_row = int(ref_hits[0])
    n = len(bim)
    with lzma.open(str(ld_dir / f"LD_{ld_block_id}.ld.gz"), "rt") as fh:
        for i, line in enumerate(fh):
            if i == ref_row:
                vals = np.fromstring(line.strip(), sep="\t")
                break
        else:
            return pd.Series(dtype=float)
    if len(vals) > n:
        vals = vals[:n]
    elif len(vals) < n:
        vals = np.pad(vals, (0, n - len(vals)), constant_values=np.nan)
    ld_r2 = pd.Series(np.square(np.abs(vals)), index=bim["variant_id"].values)
    return ld_r2.loc[bim["pos"].between(win_lo, win_hi).values]


def load_ld_r2_for_locus(chrom, lead_variant, win_lo, win_hi):
    lead_pos = parse_variant_full(lead_variant)[1]
    block_id = _ld_block_id_for_position(chrom, lead_pos)
    bim = pd.read_table(
        ld_dir / f"LD_{block_id}.bim", header=None,
        names=["chr", "variant_id", "cm", "pos", "a1", "a2"],
    )
    lead_ld_id = _resolve_ld_matrix_id(lead_variant, bim["variant_id"].tolist())
    if lead_ld_id is None:
        raise ValueError(f"Lead variant not found in LD block {block_id}")
    return _load_ld_r2_to_ref(block_id, lead_ld_id, win_lo, win_hi), coloc_to_panel(lead_variant)


def load_eqtl_nominal(tissue_id, gene_id, chrom, win_lo, win_hi):
    path = eqtl_nominal_dir / f"{tissue_id}.v11.allpairs.{chrom}.parquet"
    df = pd.read_parquet(path, columns=["variant_id", "pval_nominal"], filters=[("gene_id", "=", gene_id)])
    df["pos"] = df["variant_id"].str.split("_").str[1].astype(int)
    df = df[df["pos"].between(win_lo, win_hi)].copy()
    df["logp"] = -np.log10(df["pval_nominal"].clip(lower=1e-300))
    return df[["variant_id", "pos", "logp"]]




def load_sqtl_nominal(tissue_id, phenotype_id, chrom, win_lo, win_hi):
    """Nominal sQTL for one splicing phenotype_id (cluster:gene)."""
    path = sqtl_nominal_dir / f"{tissue_id}.v11.cis_sqtl.allpairs.{chrom}.parquet"
    df = pd.read_parquet(
        path,
        columns=["phenotype_id", "variant_id", "pval_nominal"],
        filters=[("phenotype_id", "=", phenotype_id)],
    )
    if df.empty:
        return pd.DataFrame(columns=["variant_id", "pos", "logp"])
    df = df.copy()
    df["pos"] = df["variant_id"].str.split("_").str[1].astype(int)
    df = df[df["pos"].between(win_lo, win_hi)].copy()
    df["logp"] = -np.log10(df["pval_nominal"].clip(lower=1e-300))
    return df[["variant_id", "pos", "logp"]]


def sqtl_intron_label(phenotype_id):
    """Junction ID (J1/J2/J3) from LeafCutter phenotype_id, else start–end."""
    parts = str(phenotype_id).split(":")
    if len(parts) >= 3 and parts[1].isdigit() and parts[2].isdigit():
        key = (int(parts[1]), int(parts[2]))
        return junction_label_by_intron.get(key, f"{parts[1]}–{parts[2]}")
    return str(phenotype_id)


def sqtl_phenotypes_for_gene(gene_id):
    core = gene_id.split(".")[0]
    return [p for p in sqtl_phenotypes if core in p]


def load_gene_qtl_nominal(tissue_id, gene_id, chrom, win_lo, win_hi, *, phenotypes=None, qtl_kind="eqtl"):
    """Nominal QTL −log10(p). For sQTL, pass phenotype_id(s)."""
    if qtl_kind == "eqtl":
        return load_eqtl_nominal(tissue_id, gene_id, chrom, win_lo, win_hi)
    phenos = phenotypes if phenotypes is not None else sqtl_phenotypes_for_gene(gene_id)
    if not phenos:
        raise ValueError(f"No sQTL phenotypes for {gene_id}")
    frames = []
    for ph in phenos:
        df = load_sqtl_nominal(tissue_id, ph, chrom, win_lo, win_hi)
        if len(df):
            frames.append(df)
    if not frames:
        return pd.DataFrame(columns=["variant_id", "pos", "logp"])
    if len(frames) == 1:
        return frames[0]
    cat = pd.concat(frames, ignore_index=True)
    return (
        cat.sort_values("logp", ascending=False)
        .drop_duplicates("variant_id")
        [["variant_id", "pos", "logp"]]
    )


def _tabix_chrom(chrom):
    c = str(chrom).strip()
    return c[3:] if c.lower().startswith("chr") else c


def load_gwas_nominal(gwas_id, chrom, win_lo, win_hi):
    path = gwas_nominal_dir / f"imputed_{gwas_id}.txt.gz"
    win_lo, win_hi = int(win_lo), int(win_hi)
    region = f"{_tabix_chrom(chrom)}:{win_lo}-{win_hi}"
    _log(f"  GWAS tabix fetch: {gwas_id}  {region}")
    path_s = str(path)
    with gzip.open(path_s, "rt") as fh:
        hdr = fh.readline().rstrip("\n").split("\t")
    panel_i = hdr.index("panel_variant_id")
    pos_i = hdr.index("position")
    pval_i = hdr.index("pvalue")
    tb = pysam.TabixFile(path_s)
    rows = []
    for line in tb.fetch(region=region):
        f = line.split("\t")
        rows.append((f[panel_i], int(f[pos_i]), float(f[pval_i])))
    if not rows:
        return pd.DataFrame(columns=["variant_id", "pos", "logp"])
    df = pd.DataFrame(rows, columns=["variant_id", "pos", "pvalue"])
    df["logp"] = -np.log10(df["pvalue"].clip(lower=1e-300))
    return df[["variant_id", "pos", "logp"]]


def _ld_point_colors(ld_r2_vals):
    return [ld_default_color if pd.isna(r2) else ld_cmap(ld_norm(float(r2))) for r2 in ld_r2_vals]


def _plot_lead_diamond(ax, x, y, *, s=None):
    ax.scatter(
        [x], [y], c=lead_marker_color, s=lead_marker_s if s is None else s,
        marker="D", edgecolors="none", linewidths=0, zorder=10,
    )


chrom, lead_pos, _, _ = parse_variant_full(cos_lead_variant)
win_lo = lead_pos - locus_win_half
win_hi = lead_pos + locus_win_half
lead_panel_id = coloc_to_panel(cos_lead_variant)

_log("Loading LD...")
ld_r2, _ = load_ld_r2_for_locus(chrom, cos_lead_variant, win_lo, win_hi)
ld_r2_by_panel = ld_r2.copy()
ld_r2_by_panel.index = ld_r2_by_panel.index.map(
    lambda v: v if str(v).endswith("_b38") else f"{v}_b38"
)

_log("Loading GWAS...")
gwas_df = load_gwas_nominal(gwas_primary, chrom, win_lo, win_hi)
gwas_df = gwas_df.rename(columns={"logp": "gwas_logp", "variant_id": "panel_id"})

_log("Loading sQTL + eQTL tracks...")
eqtl_tracks = []
eqtl_frames = {}
_track_i = 0
for gid in gene_ids:
    name = gene_display_name(gid)
    phenos = sqtl_phenotypes_for_gene(gid)
    if not phenos:
        raise ValueError(f"No sqtl_phenotypes for {gid}")
    for pi, ph in enumerate(phenos):
        col = f"eqtl_logp_{_track_i}"
        df = load_gene_qtl_nominal(
            tissue, gid, chrom, win_lo, win_hi, phenotypes=[ph], qtl_kind="sqtl"
        ).rename(columns={"logp": col, "variant_id": "panel_id"})
        eqtl_frames[col] = df[["panel_id", "pos", col]]
        jlab = sqtl_intron_label(ph)  # J1 / J3 from intron coords
        eqtl_tracks.append({
            "col": col,
            "gene_id": gid,
            "phenotype_id": ph,
            "label": f"{name} sQTL {jlab}",
            "name": f"{name} {jlab}",
            "biotype": gene_biotype_key(gid),
            "qtl_kind": "sqtl",
            "annotate_npc": pi == 0,  # NPC once on primary intron
        })
        _log(f"  {name} sQTL {jlab}: {len(df):,} variants")
        _track_i += 1
    # eQTL track for the same gene
    col = f"eqtl_logp_{_track_i}"
    df = load_gene_qtl_nominal(
        tissue, gid, chrom, win_lo, win_hi, qtl_kind="eqtl"
    ).rename(columns={"logp": col, "variant_id": "panel_id"})
    eqtl_frames[col] = df[["panel_id", "pos", col]]
    eqtl_tracks.append({
        "col": col,
        "gene_id": gid,
        "phenotype_id": None,
        "label": f"{name} eQTL",
        "name": f"{name} eQTL",
        "biotype": gene_biotype_key(gid),
        "qtl_kind": "eqtl",
        "annotate_npc": False,
    })
    _log(f"  {name} eQTL: {len(df):,} variants")
    _track_i += 1

# Neighbor-gene QTL tracks (same tissue)
for extra in locuscompare_extra:
    gid = extra["gene_id"]
    qkind = extra.get("qtl_kind", "eqtl")
    name = gene_display_name(gid)
    col = f"eqtl_logp_{_track_i}"
    df = load_gene_qtl_nominal(
        tissue, gid, chrom, win_lo, win_hi, qtl_kind=qkind
    ).rename(columns={"logp": col, "variant_id": "panel_id"})
    eqtl_frames[col] = df[["panel_id", "pos", col]]
    qlab = "sQTL" if qkind == "sqtl" else "eQTL"
    eqtl_tracks.append({
        "col": col,
        "gene_id": gid,
        "phenotype_id": None,
        "label": f"{name} {qlab}",
        "name": name,
        "biotype": gene_biotype_key(gid),
        "qtl_kind": qkind,
        "annotate_npc": False,
    })
    _log(f"  {name} ({qlab} extra): {len(df):,} variants")
    _track_i += 1

locus_assoc = gwas_df[["panel_id", "pos", "gwas_logp"]].copy()
for col, df in eqtl_frames.items():
    locus_assoc = locus_assoc.merge(df[["panel_id", col]], on="panel_id", how="outer")
if locus_assoc["pos"].isna().any():
    pos_map = pd.concat(
        [gwas_df[["panel_id", "pos"]]] + [df[["panel_id", "pos"]] for df in eqtl_frames.values()]
    ).drop_duplicates("panel_id")
    locus_assoc = locus_assoc.drop(columns=["pos"]).merge(pos_map, on="panel_id", how="left")

locus_assoc["ld_r2"] = locus_assoc["panel_id"].map(ld_r2_by_panel)
miss = locus_assoc["ld_r2"].isna()
if miss.any():
    alt = locus_assoc.loc[miss, "panel_id"].str.replace("_b38$", "", regex=True).map(ld_r2)
    locus_assoc.loc[miss, "ld_r2"] = alt.values

# Focal CoS VCP from Zenodo long (full_df); p-value tracks already from portal/GWAS nominals.
_vcp = (
    full_df[["variant_id", "vcp"]]
    .drop_duplicates("variant_id")
    .assign(panel_id=lambda d: d["variant_id"].map(coloc_to_panel))
    .set_index("panel_id")["vcp"]
    .astype(float)
)
locus_assoc["vcp"] = locus_assoc["panel_id"].map(_vcp).fillna(0.0)

print(f"Locus assoc table: {len(locus_assoc):,} variants  window={win_lo}-{win_hi}")
print(f"Lead in table: {(locus_assoc.panel_id == lead_panel_id).any()}")
print(f"Focal CoS: {focal_cos_id}; VCP>0 variants: {int((locus_assoc['vcp'] > 0).sum())}")
print(f"GWAS -log10(p) at lead: {locus_assoc.loc[locus_assoc.panel_id == lead_panel_id, 'gwas_logp'].values}")

### Expression metadata


In [ ]:
gtex_sample_meta = pd.read_csv(
    gtex_sample_meta_file, sep="\t", usecols=["SAMPID", "SMTSD", "SMTS"], low_memory=False
)
tissue_to_organ = gtex_sample_meta.set_index("SMTSD")["SMTS"].to_dict()
gtex_sample_map = gtex_sample_meta.drop_duplicates("SAMPID").set_index("SAMPID")["SMTSD"].to_dict()

with open(gtex_colors_json) as _fh:
    _colors = json.load(_fh)
    tissue_df = pd.DataFrame.from_dict(_colors, orient="index").reset_index(names="tissue_name")
tissue_df["tissue_color_hex"] = "#" + tissue_df["tissue_color_hex"]

dgtex_colors = pd.read_csv(dgtex_colors_file)
dgtex_site_palette = (
    dgtex_colors.drop_duplicates("Tissue Site")
    .set_index("Tissue Site")["Tissue Site colorHex"]
    .to_dict()
)
dgtex_site_palette.update({k.title(): v for k, v in list(dgtex_site_palette.items()) if k != k.title()})
dgtex_meta = pd.read_excel(dgtex_meta_file)
dgtex_sample_map = (
    dgtex_meta[dgtex_meta["EXCLUDE from Analysis Freeze"] != "Yes"]
    .drop_duplicates("Sample ID (Data Release Official ID)")
    .set_index("Sample ID (Data Release Official ID)")["Tissue Site Detail"]
    .to_dict()
)
detail_to_site = (
    dgtex_meta.drop_duplicates("Tissue Site Detail")
    .set_index("Tissue Site Detail")["Tissue Site"]
    .to_dict()
)

dgtex_site_abbrv = (
    dgtex_colors.sort_values("Tissue Site Detail")
    .drop_duplicates("Tissue Site")
    .set_index("Tissue Site")["Tissue Site Detail Abbr"]
    .to_dict()
)
_detail_abbr = dict(zip(dgtex_colors["Tissue Site Detail"], dgtex_colors["Tissue Site Detail Abbr"]))
dgtex_site_abbrv["Trachea"] = _detail_abbr.get("Trachea", "TRACHEA")
# metadata Tissue Site casing can differ from colors CSV (e.g. Adipose Tissue)
dgtex_site_abbrv.update({k.title() if k != k.title() else k: v for k, v in list(dgtex_site_abbrv.items())})
dgtex_site_abbrv["Adipose Tissue"] = dgtex_site_abbrv.get("Adipose Tissue") or dgtex_site_abbrv.get("Adipose tissue") or "ADPSBQ"

gtex_site_abbrv = (
    gtex_sample_meta.merge(tissue_df[["tissue_name", "tissue_abbrv"]], left_on="SMTSD", right_on="tissue_name")
    .drop_duplicates(["SMTS", "tissue_abbrv"])
    .sort_values("SMTSD")
    .drop_duplicates("SMTS")
    .set_index("SMTS")["tissue_abbrv"]
    .to_dict()
)
gtex_organ_palette = (
    gtex_sample_meta.merge(tissue_df[["tissue_name", "tissue_color_hex"]], left_on="SMTSD", right_on="tissue_name")
    .drop_duplicates(["SMTS", "tissue_color_hex"])
    .dropna(subset=["tissue_color_hex"])
    .groupby("SMTS")["tissue_color_hex"]
    .first()
    .to_dict()
)

site_abbrv_by_dataset = {"GTEx": gtex_site_abbrv, "dGTEx": dgtex_site_abbrv}
site_palette_by_dataset = {"GTEx": gtex_organ_palette, "dGTEx": dgtex_site_palette}
detail_to_site_by_dataset = {"GTEx": tissue_to_organ, "dGTEx": detail_to_site}

print(f"dGTEx samples: {len(dgtex_sample_map):,}; GTEx organs: {len(gtex_site_abbrv)}")


### GTEx / dGTEx organ TPM


In [ ]:
def _load_gene_tpm_site_long(tpm_path, gene_id, sample_map, detail_to_site_map, dataset):
    cols = ["dataset", "tissue_detail", "tissue_site", "tpm"]
    if gene_id is None or (isinstance(gene_id, float) and pd.isna(gene_id)):
        raise ValueError(f"gene_id required for {dataset} TPM load")
    tpm_path = Path(tpm_path)
    gene_row = pd.read_parquet(tpm_path, filters=[("Name", "==", gene_id)], engine="pyarrow")
    if gene_row.empty:
        raise ValueError(f"{gene_id} not found in {tpm_path}")
    sample_cols = [c for c in gene_row.columns if c not in {"Name", "Description"}]
    long = gene_row.melt(
        id_vars=["Name"], value_vars=sample_cols, var_name="sample_id", value_name="tpm"
    )
    long["tissue_detail"] = long["sample_id"].map(sample_map)
    long = long.dropna(subset=["tissue_detail"])
    long["tissue_site"] = long["tissue_detail"].map(lambda t: detail_to_site_map.get(t, t))
    long = long.dropna(subset=["tissue_site"])
    long["dataset"] = dataset
    return long[cols]


gid = gene_ids[0]
expr_long = pd.concat(
    [
        _load_gene_tpm_site_long(
            gtex_tpm, gid, gtex_sample_map, detail_to_site_by_dataset["GTEx"], "GTEx"
        ),
        _load_gene_tpm_site_long(
            dgtex_tpm, gid, dgtex_sample_map, detail_to_site_by_dataset["dGTEx"], "dGTEx"
        ),
    ],
    ignore_index=True,
)
print(f"{gene_display_name(gid)} TPM rows: {len(expr_long):,}")
print(expr_long.groupby("dataset")["tissue_site"].nunique().to_string())

### STAR splice-junction counts by lead-variant genotype

Built in-notebook from the WGS VCF + BA9 STAR `SJ.out.tab.gz` (protected). Coverage bigWigs are not needed for this panel.

In [ ]:
for required in (sashimi_sj_dir, sashimi_vcf, gtex_sample_meta_file):
    if not Path(required).exists():
        raise FileNotFoundError(f"missing sashimi input: {required}")

meta = pd.read_csv(
    gtex_sample_meta_file,
    sep="\t",
    usecols=["SAMPID", "SMTSD", "SMAFRZE"],
    low_memory=False,
)
ba9 = meta[(meta["SMTSD"] == ba9_smtsd) & (meta["SMAFRZE"] == "RNASEQ")].copy()
ba9["subj"] = ba9["SAMPID"].astype(str).str.extract(r"^(GTEX-[^-]+)")[0]
rows = []
for _, g in ba9.groupby("subj"):
    ok = g[g["SAMPID"].map(lambda s: (sashimi_sj_dir / f"{s}.SJ.out.tab.gz").exists())]
    if len(ok):
        rows.append(ok.iloc[0])
samples = pd.DataFrame(rows)
if samples.empty:
    raise RuntimeError("No BA9 RNA-seq samples with SJ.out.tab.gz")
print(f"BA9 RNA-seq samples with SJ: {len(samples)}")

vf = pysam.VariantFile(str(sashimi_vcf))
vcf_samples = set(vf.header.samples)
chrom, pos, ref, alt = cos_lead_variant.split(":")
pos = int(pos)
recs = list(vf.fetch(chrom, pos - 1, pos))
rec = next(r for r in recs if r.pos == pos and r.ref == ref and alt in (r.alts or ()))
gser = {}
for subj in samples["subj"]:
    if subj not in vcf_samples:
        continue
    gt = rec.samples[subj].get("GT")
    if gt is None or None in gt:
        continue
    gser[subj] = gt_order[sum(1 for a in gt if a == 1)]
gser = pd.Series(gser, name="genotype")
samples = samples[samples["subj"].isin(gser.index)].copy()
samples["genotype"] = samples["subj"].map(gser)
print(samples["genotype"].value_counts().reindex(gt_order).to_string())

want_junc = {(int(a), int(b)) for _, a, b, _ in sashimi_junctions}
junc = {g: {f"{a}-{b}": [] for _, a, b, _ in sashimi_junctions} for g in gt_order}

for row in tqdm(samples.itertuples(index=False), total=len(samples), desc="STAR SJ by genotype"):
    g = row.genotype
    counts = {ab: 0 for ab in want_junc}
    with gzip.open(sashimi_sj_dir / f"{row.SAMPID}.SJ.out.tab.gz", "rt") as fh:
        for line in fh:
            if not line.startswith("chr3\t"):
                continue
            p = line.split("\t")
            a, b = int(p[1]), int(p[2])
            if a > 35_640_500:
                break
            if (a, b) in counts:
                counts[(a, b)] = int(p[6])
    for _, a, b, _ in sashimi_junctions:
        junc[g][f"{a}-{b}"].append(counts.get((a, b), 0))

junc_keys = [f"{a}-{b}" for _, a, b, _ in sashimi_junctions]
junc_samples = np.empty((len(gt_order), len(junc_keys)), dtype=object)
for gi, g in enumerate(gt_order):
    for ji, k in enumerate(junc_keys):
        junc_samples[gi, ji] = np.asarray(junc[g][k], dtype=float)

sashimi = {
    "genotypes": np.array(gt_order, dtype=object),
    "junc_primary": np.array([p for *_, p in sashimi_junctions]),
    "junc_samples": junc_samples,
}
print(f"genotypes: {list(sashimi['genotypes'])}")

## Make figure


In [ ]:
# Junction arc colors
def _junction_arc_color(label):
    if label == "J1":
        return "#1B7F7A"
    if label == "J3":
        return "#C45C26"
    return gene_type_palette.get("novel_lncRNA", "#666666")


# --- Layout: LocusZoom | LocusCompare | GTEx/dGTEx expression ---
tracks_n = 1 + len(eqtl_tracks)
n_top = tracks_n + 2  # assoc + VCP + genes
qtl_tracks = list(eqtl_tracks)
n_cmp = len(qtl_tracks)
has_expr = len(expr_long) > 0 and expr_long["dataset"].isin(["GTEx", "dGTEx"]).any()

fig = plt.figure(figsize=(7.2, 5.0), dpi=300)
outer = fig.add_gridspec(
    1, 3,
    width_ratios=[3.05, 0.78, 0.72],
    wspace=0.35,
    left=0.06, right=0.99, top=0.96, bottom=0.06,
)
height_ratios = [0.45] * tracks_n + [0.28, 0.55, 0.22, 1.35]
left = outer[0, 0].subgridspec(n_top + 2, 1, height_ratios=height_ratios, hspace=0.22)

axes = [fig.add_subplot(left[i]) for i in range(n_top)]
ax_gap = fig.add_subplot(left[n_top])
ax_gap.set_axis_off()
zoom_gs = left[n_top + 1].subgridspec(1, 2, width_ratios=[2.55, 1.15], wspace=0.28)
axz = fig.add_subplot(zoom_gs[0, 0])
ax_star = fig.add_subplot(zoom_gs[0, 1])
axes.append(axz)
for ax in axes[1:n_top]:
    ax.sharex(axes[0])

# --- Panel a: association tracks + VCP + gene overview ---
assoc_tracks = [{"col": "gwas_logp", "label": gwas_short_label(gwas_primary)}] + [
    {"col": t["col"], "label": t["label"]} for t in eqtl_tracks
]
n_assoc = len(assoc_tracks)
assoc_axes = axes[:n_assoc]
axv = axes[n_assoc]
axg = axes[n_assoc + 1]

coset_stables = {g.split(".")[0] for g in gene_ids}
genes_in_win = gene_ann[
    (gene_ann["Chromosome"] == chrom)
    & (gene_ann["Start"] <= win_hi)
    & (gene_ann["End"] >= win_lo)
].drop_duplicates("stable")
genes_in_win = pd.concat(
    [
        genes_in_win[~genes_in_win["stable"].isin(coset_stables)],
        genes_in_win[genes_in_win["stable"].isin(coset_stables)],
    ],
    ignore_index=True,
)

# Representative-transcript exons in window
if genes_in_win.empty:
    exons_in_win = pd.DataFrame(columns=["gene_id", "transcript_id", "Start", "End"])
else:
    gids = set(genes_in_win["gene_id"])
    stables = set(genes_in_win["stable"])
    exon_df = ann[
        (ann["Feature"] == "exon")
        & (ann["Chromosome"] == chrom)
        & (ann["Start"] <= win_hi)
        & (ann["End"] >= win_lo)
    ].copy()
    exon_df = exon_df[
        exon_df["gene_id"].isin(gids)
        | exon_df["gene_id"].astype(str).str.split(".").str[0].isin(stables)
    ]
    if exon_df.empty:
        exons_in_win = exon_df
    else:
        exon_df = exon_df.assign(exon_len=lambda d: d["End"] - d["Start"])
        rep_tx = (
            exon_df.groupby(["gene_id", "transcript_id"], as_index=False)["exon_len"]
            .sum()
            .sort_values("exon_len", ascending=False)
            .drop_duplicates("gene_id")
        )
        exons_in_win = exon_df.merge(
            rep_tx[["gene_id", "transcript_id"]], on=["gene_id", "transcript_id"]
        )

sub = locus_assoc[locus_assoc["pos"].between(win_lo, win_hi)].copy()
qtl_ymax = 1.0
for tr in assoc_tracks[1:]:
    y = sub[tr["col"]]
    if y.notna().any():
        qtl_ymax = max(qtl_ymax, float(np.nanmax(y.values)))
qtl_ylim = (0, qtl_ymax * 1.12)

for i, (ax, tr) in enumerate(zip(assoc_axes, assoc_tracks)):
    y = sub[tr["col"]]
    mask = y.notna()
    ax.scatter(
        sub.loc[mask, "pos"], y[mask], s=base_marker_s,
        c=_ld_point_colors(sub.loc[mask, "ld_r2"]),
        linewidths=0, alpha=0.9, zorder=3,
    )
    lead_row = sub[sub["panel_id"] == lead_panel_id]
    if len(lead_row) and pd.notna(lead_row.iloc[0][tr["col"]]):
        _plot_lead_diamond(ax, lead_row.iloc[0]["pos"], lead_row.iloc[0][tr["col"]])
    lab = tr["label"]
    if " sQTL " in lab:
        name, rest = lab.split(" sQTL ", 1)
        ylab = f"{name}\nsQTL {rest}\n$-\\log_{{10}}(p)$"
    elif lab.endswith(" eQTL"):
        ylab = f"{lab[:-5]}\neQTL\n$-\\log_{{10}}(p)$"
    else:
        ylab = f"{lab}\n$-\\log_{{10}}(p)$"
    ax.set_ylabel(ylab, rotation=0, ha="right", va="center", labelpad=4, fontsize=4)
    ax.set_xlim(win_lo, win_hi)
    if i == 0:
        ymax = float(np.nanmax(y.values)) if mask.any() else 1.0
        ax.set_ylim(0, ymax * 1.15 if ymax > 0 else 1)
    else:
        ax.set_ylim(*qtl_ylim)
    ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=3))
    sns.despine(ax=ax, bottom=False)
    ax.spines["bottom"].set_visible(True)
    ax.spines["bottom"].set_linewidth(0.6)
    ax.tick_params(axis="x", bottom=False, labelbottom=False, length=0)
    ax.tick_params(axis="y", labelsize=4, pad=1)

ax0 = assoc_axes[0]
ax0.xaxis.set_major_locator(plt.MaxNLocator(nbins=4))
ax0.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax0.xaxis.tick_top()
ax0.tick_params(
    axis="x", which="major", top=True, labeltop=True,
    bottom=False, labelbottom=False, direction="out",
    length=1.8, width=0.45, pad=1, colors="0.15", labelsize=4,
)
ax0.spines["top"].set_visible(True)
ax0.spines["top"].set_linewidth(0.6)
ax0.text(
    1.0, 1.02, chrom, transform=ax0.transAxes,
    ha="right", va="bottom", fontsize=5, color="0.2", clip_on=False,
)
ld_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=3, label=lab)
    for c, lab in zip(ld_colors, ["0–0.2", "0.2–0.4", "0.4–0.6", "0.6–0.8", "0.8–1"])
]
ld_handles.append(Line2D(
    [0], [0], marker="D", color="w", markerfacecolor=lead_marker_color,
    markeredgecolor="none", markersize=3, label="CoS lead",
))
ax0.legend(
    handles=ld_handles, title=r"UKB · LD $r^2$",
    loc="upper right", frameon=True, fancybox=False, edgecolor="0.55",
    framealpha=0.95, fontsize=3.5, title_fontsize=4.0,
    handletextpad=0.3, labelspacing=0.15, borderpad=0.35,
    borderaxespad=0.4, markerscale=0.9,
)

cs_sub = sub[sub["vcp"] > 0]
if len(cs_sub):
    axv.scatter(
        cs_sub["pos"], cs_sub["vcp"], s=base_marker_s,
        c=_ld_point_colors(cs_sub["ld_r2"]),
        linewidths=0, alpha=0.9, zorder=3,
    )
    lead_row = cs_sub[cs_sub["panel_id"] == lead_panel_id]
    if len(lead_row):
        _plot_lead_diamond(axv, lead_row.iloc[0]["pos"], float(lead_row.iloc[0]["vcp"]))
axv.set_ylabel("VCP", rotation=0, ha="right", va="center", labelpad=4, fontsize=4)
axv.set_xlim(win_lo, win_hi)
axv.set_ylim(0, 1.05)
axv.set_yticks([0, 1.0])
sns.despine(ax=axv, bottom=False)
axv.spines["bottom"].set_visible(True)
axv.spines["bottom"].set_linewidth(0.6)
axv.tick_params(axis="x", bottom=False, labelbottom=False, length=0)
axv.tick_params(axis="y", labelsize=4, pad=1)

# Focal zoom window from transcripts / junctions / ARPP21 exons
zoom_coords = []
for tx in focal_transcripts:
    for a0, b0 in tx["exons"]:
        zoom_coords.extend([int(a0), int(b0)])
    for junc in tx["junctions"]:
        j0, j1 = junc["coords"]
        zoom_coords.extend([int(j0), int(j1)])
    for a0, b0 in tx["ncorf_exons"]:
        zoom_coords.extend([int(a0), int(b0)])
_arpp_ex = ann[
    (ann["gene_name"] == "ARPP21")
    & (ann["Feature"] == "exon")
    & (ann["Chromosome"] == chrom)
    & (ann["Start"] <= focal_zoom_win[1])
    & (ann["End"] >= focal_zoom_win[0])
]
_arpp_ivs = sorted((int(a), int(b)) for a, b in zip(_arpp_ex["Start"], _arpp_ex["End"]))
_arpp_merged = []
for a, b in _arpp_ivs:
    if not _arpp_merged or a > _arpp_merged[-1][1] + 1:
        _arpp_merged.append([a, b])
    else:
        _arpp_merged[-1][1] = max(_arpp_merged[-1][1], b)
for e0, e1 in _arpp_merged:
    zoom_coords.extend([e0, e1])
if zoom_coords:
    zoom_lo, zoom_hi = min(zoom_coords) - 120, max(zoom_coords) + 120
else:
    zoom_lo, zoom_hi = lead_pos - zoom_win_half, lead_pos + zoom_win_half
zoom_lo = max(zoom_lo, win_lo)
zoom_hi = min(zoom_hi, win_hi)

# Gene overview track (no intron/ncORF marks; those live in the zoom)
y_plus, y_minus = 0.68, 0.28
for y in (y_plus, y_minus):
    axg.axhline(y, color="0.85", lw=0.4, zorder=0)
strand_trans = blended_transform_factory(axg.transAxes, axg.transData)
axg.text(0.995, y_plus + 0.05, "+", transform=strand_trans,
         ha="right", va="bottom", fontsize=5, color="0.35", clip_on=False)
axg.text(0.995, y_minus + 0.05, "−", transform=strand_trans,
         ha="right", va="bottom", fontsize=5, color="0.35", clip_on=False)
label_slots = {"+": [], "-": []}
min_sep = 0.12 * (win_hi - win_lo)
label_pad = 0.10
for _, grow in genes_in_win.sort_values("Start").iterrows():
    strand = grow["Strand"] if grow["Strand"] in ("+", "-") else "+"
    y = y_plus if strand == "+" else y_minus
    gid = grow["gene_id"]
    stable = str(gid).split(".")[0]
    in_coset = stable in coset_stables
    color = gene_type_palette.get(gene_biotype_key(gid), "#666666")
    lw = 1.2 if in_coset else 0.45
    h = 0.18 if in_coset else 0.10
    z_body = 3 if in_coset else 1
    g0 = max(int(grow["Start"]), win_lo)
    g1 = min(int(grow["End"]), win_hi)
    if g1 <= g0:
        continue
    axg.plot([g0, g1], [y, y], color=color, lw=lw, solid_capstyle="butt", zorder=z_body)
    min_exon_w = 0.004 * (win_hi - win_lo)
    gene_exons = exons_in_win[exons_in_win["gene_id"] == gid].sort_values("Start")
    if gene_exons.empty:
        axg.add_patch(plt.Rectangle(
            (g0, y - h / 2), max(g1 - g0, min_exon_w), h,
            facecolor=color, edgecolor="none", zorder=z_body + 1, linewidth=0,
        ))
    else:
        for _, ex in gene_exons.iterrows():
            x0 = max(int(ex["Start"]), win_lo)
            x1 = min(int(ex["End"]), win_hi)
            if x1 <= x0:
                continue
            axg.add_patch(plt.Rectangle(
                (x0, y - h / 2), max(x1 - x0, min_exon_w), h,
                facecolor=color, edgecolor="none", zorder=z_body + 1, linewidth=0,
            ))
    mid = 0.5 * (g0 + g1)
    slot = 0
    for prev_mid, prev_slot in label_slots[strand]:
        if abs(mid - prev_mid) < min_sep:
            slot = max(slot, prev_slot + 1)
    label_slots[strand].append((mid, slot))
    if strand == "+":
        label_y = y + label_pad + slot * 0.11
        label_va = "bottom"
    else:
        label_y = y - label_pad
        label_va = "top"
    label = gene_display_name(gid) if in_coset else grow["gene_name"]
    label_x = mid
    label_ha = "center"
    if in_coset:
        pad = 0.012 * (win_hi - win_lo)
        label_x = min(label_x, float(zoom_lo) - pad)
        label_ha = "right"
    axg.text(
        label_x, label_y, label,
        ha=label_ha, va=label_va, fontsize=3.0, color=color,
        style="italic", clip_on=False, zorder=z_body + 3,
    )
axg.set_ylim(-0.05, 1.15)
axg.set_xlim(win_lo, win_hi)
axg.set_yticks([])
axg.set_ylabel("")
sns.despine(ax=axg, left=True, bottom=True)
axg.tick_params(axis="x", bottom=False, labelbottom=False, length=0)

# --- Panel b: transcript zoom ---
y_cos = 0.96
n_tx = len(focal_transcripts)
y_arpp = 0.74
y_txs = [0.44, 0.16][:n_tx] if n_tx <= 2 else list(np.linspace(0.46, 0.14, n_tx))
axz.axhline(y_cos, color="0.85", lw=0.4, zorder=0)
left_trans = blended_transform_factory(axz.transAxes, axz.transData)
label_x = -0.03
color = gene_type_palette.get("novel_lncRNA", "#666666")
arpp_color = gene_type_palette.get("protein_coding", "#1f78b4")
ncorf_color = "0.1"
backbone_color, backbone_lw = "0.85", 0.4
min_exon_w = 0.006 * (zoom_hi - zoom_lo)
h = 0.075

# Collapsed ARPP21 exons in zoom
_arpp_ex = ann[
    (ann["gene_name"] == "ARPP21")
    & (ann["Feature"] == "exon")
    & (ann["Chromosome"] == chrom)
    & (ann["Start"] <= zoom_hi)
    & (ann["End"] >= zoom_lo)
]
_arpp_ivs = sorted((int(a), int(b)) for a, b in zip(_arpp_ex["Start"], _arpp_ex["End"]))
arpp_exons = []
for a, b in _arpp_ivs:
    if not arpp_exons or a > arpp_exons[-1][1] + 1:
        arpp_exons.append([a, b])
    else:
        arpp_exons[-1][1] = max(arpp_exons[-1][1], b)
arpp_exons = [(a, b) for a, b in arpp_exons]
axz.text(
    label_x, y_arpp, "ARPP21 (+)", transform=left_trans,
    ha="right", va="center", fontsize=3.2, color=arpp_color,
    style="italic", clip_on=False, zorder=6,
)
axz.plot(
    [zoom_lo, zoom_hi], [y_arpp, y_arpp], color=backbone_color, lw=backbone_lw,
    solid_capstyle="butt", zorder=2,
)
for e0, e1 in arpp_exons:
    x0, x1 = max(e0, zoom_lo), min(e1, zoom_hi)
    if x1 <= x0:
        continue
    axz.add_patch(plt.Rectangle(
        (x0, y_arpp - h / 2), max(x1 - x0, min_exon_w), h,
        facecolor=arpp_color, edgecolor="none", zorder=3, linewidth=0,
    ))

for tx, y in zip(focal_transcripts, y_txs):
    axz.text(
        label_x, y, f"{tx['label']} (−)", transform=left_trans,
        ha="right", va="center", fontsize=3.2, color=color,
        style="italic", clip_on=False, zorder=6,
    )
    exons = sorted([(int(e0), int(e1)) for e0, e1 in tx["exons"]])
    g0 = max(exons[0][0], zoom_lo)
    g1 = min(exons[-1][1], zoom_hi)
    if g1 <= g0:
        continue
    axz.plot(
        [zoom_lo, zoom_hi], [y, y], color=backbone_color, lw=backbone_lw,
        solid_capstyle="butt", zorder=2,
    )
    for e0, e1 in exons:
        x0 = max(e0, zoom_lo)
        x1 = min(e1, zoom_hi)
        if x1 <= x0:
            continue
        axz.add_patch(plt.Rectangle(
            (x0, y - h / 2), max(x1 - x0, min_exon_w), h,
            facecolor=color, edgecolor="none", zorder=3, linewidth=0,
        ))
    for o0, o1 in tx["ncorf_exons"]:
        x0 = max(int(o0), zoom_lo)
        x1 = min(int(o1), zoom_hi)
        if x1 <= x0:
            continue
        axz.add_patch(plt.Rectangle(
            (x0, y - h / 2), max(x1 - x0, min_exon_w * 0.5), h,
            facecolor=ncorf_color, edgecolor="none", zorder=5, linewidth=0,
        ))
    start_pos = tx.get("start_pos")
    start_codon = tx.get("start_codon")
    if start_pos is not None and start_codon and zoom_lo <= start_pos <= zoom_hi:
        tip_y = y - h / 2
        axz.annotate(
            start_codon,
            xy=(start_pos, tip_y),
            xytext=(0, -7),
            textcoords="offset points",
            ha="center", va="top",
            fontsize=3.0, color="0.1", fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="0.1", lw=0.55, shrinkA=0, shrinkB=0),
            clip_on=False, zorder=8,
        )
    for junc in tx["junctions"]:
        j0, j1 = int(junc["coords"][0]), int(junc["coords"][1])
        x0 = max(j0, zoom_lo)
        x1 = min(j1, zoom_hi)
        if x1 <= x0:
            continue
        mid = 0.5 * (x0 + x1)
        peak_y = y + (0.12 if junc["label"] == "J3" else 0.20)
        jcolor = _junction_arc_color(junc["label"])
        axz.plot(
            [x0, mid, x1], [y, peak_y, y],
            color=jcolor, lw=0.55, solid_capstyle="round", zorder=6,
        )
        axz.text(
            mid, peak_y + 0.015, junc["label"],
            ha="center", va="bottom", fontsize=2.8, color="0.1",
            clip_on=False, zorder=7,
        )

vcp_sub = locus_assoc[
    locus_assoc["pos"].between(zoom_lo, zoom_hi) & (locus_assoc["vcp"] > 0)
]
if len(vcp_sub):
    axz.scatter(
        vcp_sub["pos"], np.full(len(vcp_sub), y_cos),
        s=base_marker_s * 1.3, c=_ld_point_colors(vcp_sub["ld_r2"]),
        linewidths=0, alpha=0.9, zorder=4,
    )
    lead_row = vcp_sub[vcp_sub["panel_id"] == lead_panel_id]
    if len(lead_row):
        lead_x = float(lead_row.iloc[0]["pos"])
        _plot_lead_diamond(axz, lead_x, y_cos)
        axz.text(
            lead_x, y_cos + 0.055, "lead CoS variant",
            ha="center", va="bottom", fontsize=3.2, color="0.1",
            clip_on=False, zorder=6,
        )

axz.set_xlim(zoom_lo, zoom_hi)
axz.set_ylim(0.0, 1.08)
axz.set_yticks([])
axz.set_ylabel("")
axz.tick_params(axis="x", bottom=False, labelbottom=False, length=0)
sns.despine(ax=axz, left=True, bottom=True)
span = zoom_hi - zoom_lo
bar = 200 if span < 3_000 else (500 if span < 8_000 else 1_000)
bx0 = zoom_hi - bar - 0.04 * span
axz.plot([bx0, bx0 + bar], [0.06, 0.06], color="0.2", lw=0.9, clip_on=False)
axz.text(
    bx0 + 0.5 * bar, 0.03,
    f"{bar} bp" if bar < 1000 else f"{bar // 1000} kb",
    ha="center", va="top", fontsize=4, color="0.25",
)
axz.legend(
    handles=[mpatches.Patch(facecolor=ncorf_color, edgecolor="none", label="ncORF")],
    loc="lower left", fontsize=3.2, frameon=False,
    handlelength=1.0, handleheight=0.7, borderpad=0.1, labelspacing=0.2,
)

# Zoom connectors: ROI band on gene track + fan into zoom
fill_rgb = mcolors.to_rgb("0.72")
line_rgb = mcolors.to_rgb("0.55")
span_alpha, line_alpha, lw_conn, n_fade = 0.16, 0.85, 0.55, 40
axg.axvspan(
    zoom_lo, zoom_hi, facecolor=fill_rgb, edgecolor="none",
    alpha=span_alpha, zorder=0, lw=0,
)
y_lo_g, y_hi_g = axg.get_ylim()
for x in (zoom_lo, zoom_hi):
    axg.plot(
        [x, x], [y_lo_g, y_hi_g], color=line_rgb, lw=lw_conn, ls="-",
        solid_capstyle="butt", zorder=0, alpha=line_alpha,
    )
_inv_fig = fig.transFigure.inverted()
ov_bl = _inv_fig.transform(axg.transData.transform((zoom_lo, y_lo_g)))
ov_br = _inv_fig.transform(axg.transData.transform((zoom_hi, y_lo_g)))
z_y0, z_y1 = axz.get_ylim()
zm_tl = _inv_fig.transform(axz.transData.transform((zoom_lo, z_y1)))
zm_tr = _inv_fig.transform(axz.transData.transform((zoom_hi, z_y1)))
for i in range(n_fade):
    t0, t1 = i / n_fade, (i + 1) / n_fade
    poly = Polygon(
        [
            (ov_bl[0] + (zm_tl[0] - ov_bl[0]) * t0, ov_bl[1] + (zm_tl[1] - ov_bl[1]) * t0),
            (ov_br[0] + (zm_tr[0] - ov_br[0]) * t0, ov_br[1] + (zm_tr[1] - ov_br[1]) * t0),
            (ov_br[0] + (zm_tr[0] - ov_br[0]) * t1, ov_br[1] + (zm_tr[1] - ov_br[1]) * t1),
            (ov_bl[0] + (zm_tl[0] - ov_bl[0]) * t1, ov_bl[1] + (zm_tl[1] - ov_bl[1]) * t1),
        ],
        transform=fig.transFigure, facecolor=fill_rgb, edgecolor="none",
        alpha=span_alpha, zorder=0, clip_on=False,
    )
    fig.add_artist(poly)
for p0, p1 in ((ov_bl, zm_tl), (ov_br, zm_tr)):
    segs = [
        [
            (p0[0] + (p1[0] - p0[0]) * (i / n_fade), p0[1] + (p1[1] - p0[1]) * (i / n_fade)),
            (p0[0] + (p1[0] - p0[0]) * ((i + 1) / n_fade), p0[1] + (p1[1] - p0[1]) * ((i + 1) / n_fade)),
        ]
        for i in range(n_fade)
    ]
    fig.add_artist(LineCollection(
        segs, colors=[(*line_rgb, line_alpha)] * n_fade, linewidths=lw_conn,
        transform=fig.transFigure, zorder=1, clip_on=False, capstyle="butt",
    ))
for i in range(n_fade):
    t0, t1 = i / n_fade, (i + 1) / n_fade
    tm = 0.5 * (t0 + t1)
    y_a = z_y1 + (z_y0 - z_y1) * t0
    y_b = z_y1 + (z_y0 - z_y1) * t1
    axz.axhspan(
        min(y_a, y_b), max(y_a, y_b), facecolor=fill_rgb, edgecolor="none",
        alpha=span_alpha * (1.0 - tm), zorder=0, lw=0,
    )
for x in (zoom_lo, zoom_hi):
    segs, cols = [], []
    for i in range(n_fade):
        t0, t1 = i / n_fade, (i + 1) / n_fade
        tm = 0.5 * (t0 + t1)
        y_a = z_y1 + (z_y0 - z_y1) * t0
        y_b = z_y1 + (z_y0 - z_y1) * t1
        segs.append([(x, y_a), (x, y_b)])
        cols.append((*line_rgb, line_alpha * (1.0 - tm)))
    axz.add_collection(LineCollection(segs, colors=cols, linewidths=lw_conn, zorder=0, capstyle="butt"))

# Re-assert top ticks after sharex siblings mute them
ax0.xaxis.tick_top()
ax0.tick_params(
    axis="x", which="major", top=True, labeltop=True,
    bottom=False, labelbottom=False, direction="out",
    length=1.8, width=0.45, pad=1, colors="0.15", labelsize=4,
)

# --- Panel c: STAR SJ violins by genotype ---
genotypes = list(sashimi["genotypes"])
junc_primary = list(sashimi["junc_primary"])
junc_samples = sashimi["junc_samples"]
junc_order = sorted(range(len(junc_primary)), key=lambda i: (0 if junc_primary[i] else 1))
positions, series, colors = [], [], []
xtick_pos, xtick_lab = [], []
pair_gap, group_gap = 0.95, 2.7
for gi, g in enumerate(genotypes):
    base = gi * group_gap
    n_g = None
    g = str(g)
    gt_lab = f"{g[0]}/{g[1]}" if len(g) == 2 else g
    for k, ji in enumerate(junc_order):
        is_pri = bool(junc_primary[ji])
        c = _junction_arc_color("J1" if is_pri else "J3")
        x = base + k * pair_gap
        vals = np.asarray(junc_samples[gi, ji], dtype=float)
        vals = vals[np.isfinite(vals)]
        positions.append(x)
        series.append(vals)
        colors.append(c)
        if n_g is None:
            n_g = int(len(vals))
    xtick_pos.append(base + 0.5 * pair_gap * (len(junc_order) - 1))
    xtick_lab.append(f"{gt_lab}\n({n_g})")
parts = ax_star.violinplot(
    series, positions=positions, widths=0.85,
    showmeans=False, showmedians=False, showextrema=False,
)
for body, c in zip(parts["bodies"], colors):
    body.set_facecolor(c)
    body.set_edgecolor("0.1")
    body.set_linewidth(0.5)
    body.set_alpha(0.85)
for x, vals in zip(positions, series):
    if len(vals) == 0:
        continue
    q1, med, q3 = np.percentile(vals, [25, 50, 75])
    ax_star.plot([x, x], [q1, q3], color="0.1", lw=1.0, solid_capstyle="butt", zorder=5)
    ax_star.scatter(
        [x], [med], s=1.2, color="white", edgecolors="0.1",
        linewidths=0.2, zorder=6,
    )
ax_star.set_xticks(xtick_pos)
ax_star.set_xticklabels(xtick_lab, fontsize=4.0)
ax_star.set_xlim(positions[0] - 0.9, positions[-1] + 0.9)
ax_star.set_ylabel("STAR SJ count", fontsize=4)
ax_star.tick_params(axis="y", labelsize=4, length=1.8, width=0.45, pad=1)
ax_star.tick_params(axis="x", labelsize=4, length=1.8, width=0.45, pad=1)
ax_star.yaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=5))
c_j1, c_j3 = _junction_arc_color("J1"), _junction_arc_color("J3")
ax_star.legend(
    handles=[
        mpatches.Patch(facecolor=c_j1, edgecolor="0.1", linewidth=0.5, label="J1"),
        mpatches.Patch(facecolor=c_j3, edgecolor="0.1", linewidth=0.5, label="J3"),
    ],
    loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=2,
    frameon=False, fontsize=4.0, handlelength=0.9,
    handletextpad=0.3, columnspacing=0.9, borderaxespad=0.0,
)
sns.despine(ax=ax_star)
ax_star.spines["left"].set_linewidth(0.7)
ax_star.spines["bottom"].set_linewidth(0.7)

# --- Panel d: LocusCompare stack ---
xmax = ymax = 1.0
for tr in qtl_tracks:
    lc = locus_assoc[locus_assoc["gwas_logp"].notna() & locus_assoc[tr["col"]].notna()]
    if len(lc):
        xmax = max(xmax, float(np.nanmax(lc[tr["col"]].values)))
        ymax = max(ymax, float(np.nanmax(lc["gwas_logp"].values)))
xlim, ylim = (0, xmax * 1.05), (0, ymax * 1.05)
gwas_lab = gwas_short_label(gwas_primary)
right = outer[0, 1].subgridspec(n_cmp, 1, hspace=0.5)
cmp_axes = []
for i, tr in enumerate(qtl_tracks):
    ax = fig.add_subplot(right[i])
    lc = locus_assoc[locus_assoc["gwas_logp"].notna() & locus_assoc[tr["col"]].notna()].copy()
    is_lead = lc["panel_id"].to_numpy() == lead_panel_id
    sub_bg, sub_ld = lc.loc[~is_lead], lc.loc[is_lead]
    if len(sub_bg):
        ax.scatter(
            sub_bg[tr["col"]], sub_bg["gwas_logp"], s=base_marker_s * 0.7,
            c=_ld_point_colors(sub_bg["ld_r2"]), linewidths=0, alpha=0.85, zorder=5,
        )
    if len(sub_ld):
        r = sub_ld.iloc[0]
        _plot_lead_diamond(ax, r[tr["col"]], r["gwas_logp"])
    x_lab = tr.get("label") or tr["name"]
    ax.set_xlabel(fr"{x_lab} $-\log_{{10}}(P)$", fontsize=4.0)
    ax.set_ylabel(fr"{gwas_lab} $-\log_{{10}}(P)$", fontsize=4.0)
    ax.xaxis.set_major_locator(plt.MaxNLocator(nbins=3))
    ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=3))
    ax.tick_params(labelsize=3.5)
    # Annotate CoS NPC only when this phenotype shares a CoS with GWAS
    ph = tr.get("phenotype_id")
    shared = None
    if ph is not None and not full_df.empty:
        best = None
        for cs, g in full_df.groupby("cos_id"):
            if not str(cs).startswith("cos"):
                continue
            pheno = str(g.phenotype_id.iloc[0])
            parts = pheno.split(";")
            if gwas_primary not in pheno:
                continue
            qi = next(
                (j for j, p in enumerate(parts) if p == ph or p.endswith(ph) or ph in p),
                None,
            )
            if qi is None or pd.isna(g.cos_npc.iloc[0]):
                continue
            cos_npc = float(g.cos_npc.iloc[0])
            npcs = [float(x) for x in str(g.npc_outcomes.iloc[0]).split(";")]
            qtl_npc = npcs[qi] if qi < len(npcs) else np.nan
            if best is None or cos_npc > best[0]:
                best = (cos_npc, qtl_npc)
        shared = best
    if shared is not None:
        cos_npc, qtl_npc = shared
        lines = [rf"$CoS_{{NPC}}$={cos_npc:.2f}"]
        if pd.notna(qtl_npc):
            lines.append(rf"$NPC_{{out}}$={float(qtl_npc):.2f}")
        ax.text(
            0.97, 0.03, "\n".join(lines), transform=ax.transAxes,
            ha="right", va="bottom", fontsize=3.2, color="0.15",
            linespacing=1.15,
            bbox=dict(boxstyle="round,pad=0.18", facecolor="white", edgecolor="0.85", lw=0.35),
            zorder=8,
        )
    sns.despine(ax=ax)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    cmp_axes.append(ax)

# --- Panel e: GTEx / dGTEx organ TPM ---
expr_axes = []
if has_expr:
    group_col = "tissue_site"
    df = expr_long[expr_long["dataset"].isin(["GTEx", "dGTEx"])].copy()
    gene_lab = gene_display_name(gene_ids[0])
    med_max = float(df.groupby(["dataset", group_col])["tpm"].median().max()) if len(df) else 1.0
    xmax_tpm = max(med_max * 2.0, 1.0)
    expr_datasets = [d for d in ["GTEx", "dGTEx"] if (df["dataset"] == d).any()]
    height_ratios_e = [
        max(int(df.loc[df["dataset"] == d, group_col].nunique()), 1) for d in expr_datasets
    ]
    expr_gs = outer[0, 2].subgridspec(
        len(expr_datasets), 1, height_ratios=height_ratios_e, hspace=0.04,
    )
    expr_axes = [fig.add_subplot(expr_gs[i, 0]) for i in range(len(expr_datasets))]
    for ax in expr_axes[1:]:
        ax.sharex(expr_axes[0])
    ylabel_by = {"GTEx": "GTEx organ", "dGTEx": "dGTEx organ"}
    for ax, dataset in zip(expr_axes, expr_datasets):
        esub = df[df["dataset"] == dataset].copy()
        sites = sorted(esub[group_col].dropna().unique().tolist())
        abbr_map = site_abbrv_by_dataset.get(dataset, {})
        site_pal = site_palette_by_dataset.get(dataset, {})
        y_order, y_labels, palette = [], [], {}
        for site in sites:
            key = f"{dataset}::{site}"
            y_order.append(key)
            lab = abbr_map.get(site) or next(
                (abbr_map[k] for k in abbr_map if str(k).lower() == str(site).lower()),
                site,
            )
            y_labels.append(str(lab))
            col = site_pal.get(site) or next(
                (site_pal[k] for k in site_pal if str(k).lower() == str(site).lower()),
                "#cccccc",
            )
            if isinstance(col, str) and col and not str(col).startswith("#"):
                col = "#" + str(col)
            palette[key] = col
        plot_df = esub.copy()
        plot_df["y_key"] = dataset + "::" + plot_df[group_col].astype(str)
        plot_df = plot_df[plot_df["y_key"].isin(y_order)]
        plot_df["tpm"] = plot_df["tpm"].clip(upper=xmax_tpm)
        sns.boxplot(
            data=plot_df, y="y_key", x="tpm", order=y_order,
            hue="y_key", hue_order=y_order, palette=palette,
            dodge=False, ax=ax, linewidth=0.3, saturation=1.0, orient="h",
            fliersize=0.5, width=0.7,
            flierprops=dict(marker="o", markersize=0.5, markeredgewidth=0, linestyle="none"),
        )
        for flier in [ln for ln in ax.get_lines() if ln.get_linestyle() == "None" and ln.get_marker() == "o"]:
            ydata = flier.get_ydata()
            if len(ydata) == 0:
                continue
            yi = int(round(ydata[0]))
            if 0 <= yi < len(y_order):
                flier.set_markerfacecolor(palette[y_order[yi]])
                flier.set_markeredgecolor(palette[y_order[yi]])
        ax.set_ylim(len(y_order) - 0.5, -0.5)
        ax.set_xlim(0, xmax_tpm)
        ax.set_yticks(range(len(y_order)))
        ax.set_yticklabels(y_labels, fontsize=2.4)
        ax.set_ylabel(ylabel_by.get(dataset, dataset), fontsize=4.0, labelpad=1)
        ax.tick_params(axis="y", length=0, pad=1)
        ax.spines[["top", "right"]].set_visible(False)
        if ax.get_legend() is not None:
            ax.get_legend().remove()
        if ax is expr_axes[-1]:
            ax.set_xlabel(f"{gene_lab} TPM", fontsize=4.5)
            ax.tick_params(axis="x", labelsize=3.5, length=1.5, width=0.4, pad=1)
        else:
            ax.set_xlabel("")
            ax.tick_params(axis="x", labelbottom=False, length=1.5, width=0.4)

# Panel letters
_inv = fig.transFigure.inverted()
_letter_specs = [
    (axes[0], "a", -0.08, 1.18),
    (axz, "b", -0.10, 1.22),
    (ax_star, "c", -0.10, 1.22),
    (cmp_axes[0], "d", -0.18, 1.08),
]
if expr_axes:
    _letter_specs.append((expr_axes[0], "e", -0.35, 1.05))
for ax, label, x_off, y_ax in _letter_specs:
    _x = _inv.transform(ax.transAxes.transform((x_off, 0)))[0]
    _y = _inv.transform(ax.transAxes.transform((0, y_ax)))[1]
    fig.text(_x, _y, label, fontweight="bold", va="bottom", ha="right", fontsize=9)

plt.show()

## Save figure


In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / "figure_5.pdf", bbox_inches="tight")
fig.savefig(fig_dir / "figure_5.png", bbox_inches="tight")
print(f"Wrote {fig_dir / 'figure_5.pdf'}")
print(f"Wrote {fig_dir / 'figure_5.png'}")

## Print stats


In [ ]:
print(f"CoS lead variant: {cos_lead_variant}")

# Normalized probability of colocalization of the focal CoS
focal_cos_npc = float(full_df.loc[full_df["cos_id"] == focal_cos_id, "cos_npc"].iloc[0])
print(f"CoS npc: {focal_cos_npc:.2f}")

# Samples per genotype at the lead variant
for gi, g in enumerate(sashimi["genotypes"]):
    g = str(g)
    gt_label = f"{g[0]}/{g[1]}" if len(g) == 2 else g
    n_g = int(np.isfinite(np.asarray(sashimi["junc_samples"][gi, 0], dtype=float)).sum())
    print(f"genotype {gt_label}: n = {n_g}")

# Distance from lead to J3 acceptor (minus strand: acceptor = lower intron coordinate)
j3_intron = next(k for k, v in junction_label_by_intron.items() if v == "J3")
print(f"CoS lead distance to J3 splice acceptor: {abs(lead_pos - min(j3_intron))} bp")

# Ribo-seq ncORF codon length on the canonical junction transcript (ENST00000730226)
_tx = next(t for t in focal_transcripts if t["transcript_id"].startswith("ENST00000730226"))
_orf_bp = sum(b - a + 1 for a, b in _tx["ncorf_exons"])  # inclusive exon coords
print(f"ncORF length: {_orf_bp // 3} codons ({_orf_bp} bp)")
